# Mosquito Multi-Head Classifier (EfficientNet-B0)
Species + Sex, two-stage freeze/unfreeze training, Comet ML tracking.

**Runtime:** Colab, A100 GPU (Runtime > Change runtime type > A100)

**Assumes you already have:**
- Your unzipped CDC/MR4 mosquito image folder
- `labels.csv` + `class_maps.json` built by the earlier labeling script

If you haven't built those yet, run the labeling script from before first.


## 1. Install dependencies

In [ ]:
!pip install -q comet_ml albumentations torchmetrics


## 2. Comet ML setup + verification
Set your API key ONCE here. This is the single source of truth used for the whole notebook.

**Rotate your key on comet.com before running this if you've ever pasted it in plaintext anywhere.**


In [ ]:
import os

os.environ["COMET_API_KEY"]      = "PASTE_YOUR_COMET_API_KEY_HERE"   # <-- replace
os.environ["COMET_PROJECT_NAME"] = "vectorcam-mosquito"

import comet_ml
comet_ml.login()

# Hard check: does a real experiment get created?
_test = comet_ml.Experiment(project_name="vectorcam-mosquito")
print("COMET OK ->", _test.url)   # must print a real comet.com URL
_test.end()


## 3. Imports (torch imported AFTER comet, so the integration hooks in)

In [ ]:
import json
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchmetrics.classification import MulticlassAccuracy, MulticlassConfusionMatrix

import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 4. Point at your dataset
Set `ROOT` to wherever you unzipped your CDC/MR4 folder (the one containing `labels.csv`).


In [ ]:
ROOT = "/content/mosquito_cdc"   # <-- change if yours is named differently

LABELS_CSV = os.path.join(ROOT, "labels.csv")
CLASS_MAPS_JSON = os.path.join(ROOT, "class_maps.json")

df = pd.read_csv(LABELS_CSV)
with open(CLASS_MAPS_JSON) as f:
    class_maps = json.load(f)

species_map = class_maps["species"]
sex_map     = class_maps["sex"]

n_species = len(species_map)
n_sex     = len(sex_map)

print(f"Loaded {len(df)} rows")
print(f"Species classes ({n_species}): {list(species_map.keys())}")
print(f"Sex classes ({n_sex}): {list(sex_map.keys())}")
print(df["split"].value_counts())


## 5. Dataset + augmentation
Heavier augmentation since we only have ~500 training images. Flips both directions (no canonical
"up" for a mosquito on a slide), rotation, color jitter, and modest crop/scale jitter.


In [ ]:
IMG_SIZE = 224

train_tfms = A.Compose([
    A.Resize(256, 256),
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.75, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=25, p=0.7),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.5),
    A.GaussNoise(p=0.15),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

eval_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


class MosquitoDataset(Dataset):
    def __init__(self, dataframe, root, species_map, sex_map, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.root = root
        self.species_map = species_map
        self.sex_map = sex_map
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.root, row["filepath"])
        img = np.array(Image.open(path).convert("RGB"))

        if self.transform:
            img = self.transform(image=img)["image"]

        species_id = int(row["species_id"])
        sex_id     = int(row["sex_id"])

        return img, {"species": species_id, "sex": sex_id}


train_df = df[df["split"] == "train"]
val_df   = df[df["split"] == "val"]
test_df  = df[df["split"] == "test"]

train_ds = MosquitoDataset(train_df, ROOT, species_map, sex_map, transform=train_tfms)
val_ds   = MosquitoDataset(val_df,   ROOT, species_map, sex_map, transform=eval_tfms)
test_ds  = MosquitoDataset(test_df,  ROOT, species_map, sex_map, transform=eval_tfms)

BATCH_SIZE = 32

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")


## 6. Model: EfficientNet-B0, two heads (species + sex)

In [ ]:
class MosquitoMultiHead(nn.Module):
    def __init__(self, n_species, n_sex, freeze_backbone=True):
        super().__init__()
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.trunk = backbone.features
        self.pool  = nn.AdaptiveAvgPool2d(1)
        feat_dim   = backbone.classifier[1].in_features  # 1280

        if freeze_backbone:
            for p in self.trunk.parameters():
                p.requires_grad = False

        self.species_head = nn.Sequential(nn.Dropout(0.3), nn.Linear(feat_dim, n_species))
        self.sex_head     = nn.Sequential(nn.Dropout(0.3), nn.Linear(feat_dim, n_sex))

    def forward(self, x):
        feats = self.pool(self.trunk(x)).flatten(1)
        return {"species": self.species_head(feats), "sex": self.sex_head(feats)}


def set_backbone_trainable(model, block_indices):
    """Unfreeze specific blocks of features (EfficientNet-B0 has 9 top-level blocks: 0-8)."""
    for name, p in model.trunk.named_parameters():
        block_num = int(name.split(".")[0])
        p.requires_grad = block_num in block_indices


model = MosquitoMultiHead(n_species=n_species, n_sex=n_sex, freeze_backbone=True).to(device)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Total params:     {sum(p.numel() for p in model.parameters()):,}")


## 7. Loss + metrics
Simple sum of two cross-entropies. (Strain head dropped — every species in this dataset has only
one strain, so there was nothing for a third head to learn.)


In [ ]:
def multitask_loss(preds, targets, w_species=1.0, w_sex=1.0):
    loss_species = nn.functional.cross_entropy(preds["species"], targets["species"])
    loss_sex     = nn.functional.cross_entropy(preds["sex"], targets["sex"])
    total = w_species * loss_species + w_sex * loss_sex
    return total, {"species": loss_species.item(), "sex": loss_sex.item()}


species_acc_metric = MulticlassAccuracy(num_classes=n_species, average="macro").to(device)
sex_acc_metric     = MulticlassAccuracy(num_classes=n_sex,     average="macro").to(device)


## 8. Train / eval loop helpers

In [ ]:
def move_targets(targets, device):
    return {k: v.to(device) for k, v in targets.items()}


def run_epoch(model, loader, optimizer=None, experiment=None, epoch=None, stage=""):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    species_acc_metric.reset()
    sex_acc_metric.reset()

    running_loss, running_species_loss, running_sex_loss = 0.0, 0.0, 0.0
    n_batches = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for imgs, targets in loader:
            imgs = imgs.to(device)
            targets = move_targets(targets, device)

            preds = model(imgs)
            loss, loss_parts = multitask_loss(preds, targets)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            running_loss += loss.item()
            running_species_loss += loss_parts["species"]
            running_sex_loss += loss_parts["sex"]
            n_batches += 1

            species_acc_metric.update(preds["species"].argmax(1), targets["species"])
            sex_acc_metric.update(preds["sex"].argmax(1), targets["sex"])

    avg_loss = running_loss / n_batches
    avg_species_loss = running_species_loss / n_batches
    avg_sex_loss = running_sex_loss / n_batches
    species_acc = species_acc_metric.compute().item()
    sex_acc = sex_acc_metric.compute().item()

    split_name = "train" if is_train else "val"
    print(f"[{stage}] epoch {epoch} {split_name} | loss {avg_loss:.4f} "
          f"| species_acc {species_acc:.4f} | sex_acc {sex_acc:.4f}")

    if experiment is not None:
        experiment.log_metrics({
            f"{split_name}_loss": avg_loss,
            f"{split_name}_species_loss": avg_species_loss,
            f"{split_name}_sex_loss": avg_sex_loss,
            f"{split_name}_species_acc": species_acc,
            f"{split_name}_sex_acc": sex_acc,
        }, epoch=epoch)

    return avg_loss, species_acc, sex_acc


## 9. Stage 1 — train heads only, backbone frozen

In [ ]:
experiment = comet_ml.Experiment(project_name="vectorcam-mosquito")
experiment.set_name("mosquito_effnetb0_stage1_frozen")
experiment.log_parameters({
    "model": "efficientnet_b0",
    "stage": "1_frozen_backbone",
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "n_species": n_species,
    "n_sex": n_sex,
})

STAGE1_EPOCHS = 20
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STAGE1_EPOCHS)

best_val_loss = float("inf")
os.makedirs(os.path.join(ROOT, "checkpoints"), exist_ok=True)
STAGE1_CKPT = os.path.join(ROOT, "checkpoints", "stage1_best.pt")

for epoch in range(1, STAGE1_EPOCHS + 1):
    run_epoch(model, train_loader, optimizer=optimizer, experiment=experiment, epoch=epoch, stage="stage1")
    val_loss, val_species_acc, val_sex_acc = run_epoch(
        model, val_loader, optimizer=None, experiment=experiment, epoch=epoch, stage="stage1"
    )
    scheduler.step()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), STAGE1_CKPT)
        print(f"  -> saved new best checkpoint (val_loss={val_loss:.4f})")

experiment.end()
print("Stage 1 done. Best checkpoint:", STAGE1_CKPT)


## 10. Stage 2 — unfreeze last blocks, fine-tune at lower LR
EfficientNet-B0's `features` module has blocks 0-8. We unfreeze the last three (6, 7, 8) so the
highest-level features adapt to mosquito morphology, while the low-level texture/edge features
(which transfer fine from ImageNet) stay frozen.


In [ ]:
model.load_state_dict(torch.load(STAGE1_CKPT))
set_backbone_trainable(model, block_indices={6, 7, 8})

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

experiment2 = comet_ml.Experiment(project_name="vectorcam-mosquito")
experiment2.set_name("mosquito_effnetb0_stage2_finetune")
experiment2.log_parameters({
    "model": "efficientnet_b0",
    "stage": "2_finetune_last_blocks",
    "unfrozen_blocks": "6,7,8",
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
})

STAGE2_EPOCHS = 20
optimizer2 = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4
)
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=STAGE2_EPOCHS)

best_val_loss2 = float("inf")
STAGE2_CKPT = os.path.join(ROOT, "checkpoints", "stage2_best.pt")

for epoch in range(1, STAGE2_EPOCHS + 1):
    run_epoch(model, train_loader, optimizer=optimizer2, experiment=experiment2, epoch=epoch, stage="stage2")
    val_loss, val_species_acc, val_sex_acc = run_epoch(
        model, val_loader, optimizer=None, experiment=experiment2, epoch=epoch, stage="stage2"
    )
    scheduler2.step()

    if val_loss < best_val_loss2:
        best_val_loss2 = val_loss
        torch.save(model.state_dict(), STAGE2_CKPT)
        print(f"  -> saved new best checkpoint (val_loss={val_loss:.4f})")

experiment2.end()
print("Stage 2 done. Best checkpoint:", STAGE2_CKPT)


## 11. Final evaluation on held-out test set

In [ ]:
model.load_state_dict(torch.load(STAGE2_CKPT))
model.eval()

species_acc_metric.reset()
sex_acc_metric.reset()

species_cm = MulticlassConfusionMatrix(num_classes=n_species).to(device)
sex_cm     = MulticlassConfusionMatrix(num_classes=n_sex).to(device)

with torch.no_grad():
    for imgs, targets in test_loader:
        imgs = imgs.to(device)
        targets = move_targets(targets, device)
        preds = model(imgs)

        species_acc_metric.update(preds["species"].argmax(1), targets["species"])
        sex_acc_metric.update(preds["sex"].argmax(1), targets["sex"])
        species_cm.update(preds["species"].argmax(1), targets["species"])
        sex_cm.update(preds["sex"].argmax(1), targets["sex"])

print(f"TEST species macro-acc: {species_acc_metric.compute().item():.4f}")
print(f"TEST sex macro-acc:     {sex_acc_metric.compute().item():.4f}")


## 12. Confusion matrices

In [ ]:
import seaborn as sns

species_names = list(species_map.keys())
sex_names = list(sex_map.keys())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(species_cm.compute().cpu().numpy(), annot=True, fmt="d", cmap="Blues",
            xticklabels=species_names, yticklabels=species_names, ax=axes[0])
axes[0].set_title("Species Confusion Matrix")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
plt.setp(axes[0].get_xticklabels(), rotation=45, ha="right")

sns.heatmap(sex_cm.compute().cpu().numpy(), annot=True, fmt="d", cmap="Greens",
            xticklabels=sex_names, yticklabels=sex_names, ax=axes[1])
axes[1].set_title("Sex Confusion Matrix")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")

plt.tight_layout()
plt.savefig(os.path.join(ROOT, "confusion_matrices.png"), dpi=150)
plt.show()


## 13. Save + download final model

In [ ]:
FINAL_MODEL_PATH = os.path.join(ROOT, "mosquito_multihead_effnetb0_final.pt")
torch.save({
    "model_state_dict": model.state_dict(),
    "species_map": species_map,
    "sex_map": sex_map,
    "img_size": IMG_SIZE,
}, FINAL_MODEL_PATH)

print("Saved:", FINAL_MODEL_PATH)

from google.colab import files
files.download(FINAL_MODEL_PATH)


## Notes
- **Comet dashboard**: check the printed experiment URL after stage 1 and stage 2 start — that's
  where the loss curves, per-epoch metrics, and confusion matrix live.
- **Class imbalance**: `anopheles_atroparvus` has zero male images in this dataset. The sex head
  will never see a male atroparvus during training or eval — this is a data limitation, not a bug.
- **Next step**: once you get real field data with abdomen status, add a third head with the same
  masked-loss pattern (label = -1 for not-applicable rows, skip those rows in that head's loss term).
